# Week 11 Puzzles — Web App Development with Streamlit

> **NS5116 Computational Neuroscience — Spring 2026**

This notebook is divided into two parts:

- **Part 1 — Guided Practice (Puzzles 1–10):** Each puzzle includes a worked solution.
- **Part 2 — Independent Practice (Puzzles 11–20):** Write your own solution in the empty code cells.

> **Note:** Streamlit apps run via `streamlit run app.py`, not inside Jupyter. These puzzles practise the pandas, layout, and logic skills you need for building Streamlit dashboards. Where relevant, the puzzle simulates widget return values as plain Python variables.

---

## Part 1 — Guided Practice (with solutions)

### Puzzle 1 — Filter a DataFrame by a Widget Value

Simulate a Streamlit `selectbox` that returns a selected county name. Filter a DataFrame to show only rows for that county.

Create a DataFrame with columns `county`, `station`, `pm25`, then filter by `selected_county = "Taipei"`.

In [ ]:
import pandas as pd

df = pd.DataFrame({
    "county":  ["Taipei", "Taipei", "Kaohsiung", "Kaohsiung", "Taichung"],
    "station": ["Songshan", "Banqiao", "Zuoying", "Xiaogang", "Xitun"],
    "pm25":    [22, 25, 35, 40, 28],
})

# Simulate selectbox return value
selected_county = "Taipei"

filtered = df[df["county"] == selected_county]
print(f"Showing data for {selected_county}:")
print(filtered.to_string(index=False))

### Puzzle 2 — Filter by a Slider Range

Simulate a Streamlit `slider` that returns a `(min_val, max_val)` tuple. Filter the DataFrame to keep only rows with PM2.5 within that range.

Use `selected_range = (20, 30)`.

In [ ]:
# Simulate slider return value
selected_range = (20, 30)

filtered = df[(df["pm25"] >= selected_range[0]) & (df["pm25"] <= selected_range[1])]
print(f"PM2.5 between {selected_range[0]} and {selected_range[1]}:")
print(filtered.to_string(index=False))

### Puzzle 3 — Compute Metrics for `st.metric`

Streamlit `st.metric(label, value, delta)` shows a number with an optional change indicator.

Given two DataFrames (today vs. yesterday), compute the current PM2.5 average and the change (delta). Print values that would go into `st.metric()`.

In [ ]:
today = pd.DataFrame({"station": ["A", "B", "C"], "pm25": [22, 28, 25]})
yesterday = pd.DataFrame({"station": ["A", "B", "C"], "pm25": [30, 35, 20]})

current_avg = today["pm25"].mean()
prev_avg    = yesterday["pm25"].mean()
delta       = current_avg - prev_avg

print(f"st.metric(")
print(f"    label = 'Average PM2.5',")
print(f"    value = '{current_avg:.1f} µg/m³',")
print(f"    delta = '{delta:+.1f}',")
print(f")")
print(f"\n{'↓ Improved' if delta < 0 else '↑ Worsened'}")

### Puzzle 4 — Load CSV with `parse_dates`

Create a CSV string with date and PM2.5 columns, load it with `pd.read_csv()`, and use `parse_dates` to automatically convert the date column to `datetime64`.

Verify the dtype is datetime.

In [ ]:
import io

csv_data = """date,station,pm25
2024-01-01,Taipei,22
2024-01-02,Taipei,25
2024-01-03,Taipei,18
2024-01-01,Kaohsiung,35
2024-01-02,Kaohsiung,30
2024-01-03,Kaohsiung,38"""

df = pd.read_csv(io.StringIO(csv_data), parse_dates=["date"])

print(df.dtypes)
print()
print(df.head())

### Puzzle 5 — Simulate Column Layout Logic

Streamlit `st.columns(n)` creates side-by-side columns. Simulate the logic of distributing 4 metric cards across 2 rows of 2 columns.

Given 4 metrics, pair them into rows and print which metric goes where.

In [ ]:
metrics = [
    ("Avg PM2.5",   25.0,  -3.2),
    ("Max PM2.5",   42.0,  +5.0),
    ("Stations",    76,     0),
    ("Unhealthy %", 12.5, -2.1),
]

n_cols = 2
for row_start in range(0, len(metrics), n_cols):
    row = metrics[row_start:row_start + n_cols]
    row_num = row_start // n_cols + 1
    for col_idx, (label, value, delta) in enumerate(row):
        print(f"  Row {row_num}, Col {col_idx+1}: st.metric('{label}', {value}, {delta:+})")
    print()

### Puzzle 6 — Prepare GroupBy Data for a Bar Chart

Given a DataFrame of PM2.5 readings, group by county, compute the mean, sort descending, and prepare the data in the format needed for a bar chart.

Print the result as a clean table.

In [ ]:
df = pd.DataFrame({
    "county": ["Taipei"] * 3 + ["Kaohsiung"] * 3 + ["Taichung"] * 3,
    "pm25":   [22, 25, 18, 35, 30, 38, 28, 32, 26],
})

county_avg = (
    df.groupby("county")["pm25"]
    .mean()
    .reset_index()
    .sort_values("pm25", ascending=False)
    .rename(columns={"pm25": "mean_pm25"})
)

print(county_avg.to_string(index=False))

### Puzzle 7 — Simulate `@st.cache_data` Behaviour

Streamlit's `@st.cache_data` runs a function only once and caches the result. Simulate this pattern using a plain Python dict as a cache.

Call the cached function 3 times and show that the expensive computation runs only once.

In [ ]:
import time

_cache = {}

def cached_load_data(source):
    """Simulate @st.cache_data — cache by source key."""
    if source in _cache:
        print(f"  Cache HIT for '{source}'")
        return _cache[source]

    print(f"  Cache MISS for '{source}' — loading...")
    time.sleep(0.5)   # simulate slow load
    data = pd.DataFrame({"station": ["A", "B"], "pm25": [22, 35]})
    _cache[source] = data
    return data


for i in range(3):
    print(f"Call {i+1}:")
    df = cached_load_data("epa_api")

# Clear cache
_cache.clear()

### Puzzle 8 — Build a Sidebar Filter Chain

Simulate a Streamlit sidebar with cascading filters: first select a county, then a station within that county.

Given a DataFrame, show how the station list depends on the county selection.

In [ ]:
df = pd.DataFrame({
    "county":  ["Taipei", "Taipei", "Kaohsiung", "Kaohsiung", "Taichung"],
    "station": ["Songshan", "Banqiao", "Zuoying", "Xiaogang", "Xitun"],
    "pm25":    [22, 25, 35, 40, 28],
})

# Simulate sidebar selections
counties = sorted(df["county"].unique())
print(f"County options: {counties}")

selected_county = "Kaohsiung"   # simulates st.sidebar.selectbox
stations = sorted(df[df["county"] == selected_county]["station"].unique())
print(f"Station options for {selected_county}: {stations}")

selected_station = stations[0]   # simulates st.sidebar.selectbox
filtered = df[(df["county"] == selected_county) & (df["station"] == selected_station)]
print(f"\nFiltered data:")
print(filtered.to_string(index=False))

### Puzzle 9 — Parse and Validate `requirements.txt`

Write a function `check_requirements(filepath, required_packages)` that reads a `requirements.txt` and checks whether all required packages are listed.

Return missing packages. Test with a requirements file that is missing `plotly`.

In [ ]:
import os
import re

def check_requirements(filepath, required_packages):
    """Check that all required packages appear in requirements.txt.

    Returns:
        list[str]: Missing package names.
    """
    with open(filepath, encoding="utf-8") as f:
        lines = f.readlines()

    installed = set()
    for line in lines:
        line = line.strip()
        if not line or line.startswith("#"):
            continue
        name = re.split(r"[><=!]", line)[0].strip().lower()
        installed.add(name)

    missing = [p for p in required_packages if p.lower() not in installed]
    return missing


# Create a test requirements.txt
with open("test_req.txt", "w") as f:
    f.write("streamlit>=1.33\npandas>=2.0\n")

missing = check_requirements("test_req.txt", ["streamlit", "pandas", "plotly"])
print(f"Missing packages: {missing}")

os.remove("test_req.txt")

### Puzzle 10 — Compute a Summary Table for Display

Build a summary DataFrame suitable for `st.dataframe()`: group by county, compute mean, max, and count of PM2.5, then format numbers to 1 decimal place.

Print the formatted table.

In [ ]:
df = pd.DataFrame({
    "county": ["Taipei"] * 4 + ["Kaohsiung"] * 4 + ["Taichung"] * 3,
    "pm25":   [22, 25, 18, 20, 35, 30, 38, 42, 28, 32, 26],
})

summary = (
    df.groupby("county")["pm25"]
    .agg(["mean", "max", "count"])
    .rename(columns={"mean": "Avg PM2.5", "max": "Max PM2.5", "count": "Readings"})
    .sort_values("Avg PM2.5", ascending=False)
)

# Format for display
summary["Avg PM2.5"] = summary["Avg PM2.5"].round(1)

print(summary.to_string())

---

## Part 2 — Independent Practice (write your own solutions)

### Puzzle 11 — Multi-Select Filter

Simulate `st.multiselect()` returning a list of selected counties. Filter the DataFrame to keep rows matching any of the selected counties.

Test with `selected = ["Taipei", "Taichung"]`.

In [ ]:
# Your solution here


### Puzzle 12 — Date Range Filter

Simulate `st.date_input()` returning a start and end date. Filter a DataFrame with a datetime column to keep only rows within the date range.

Create a DataFrame spanning January 2024 and filter to Jan 10–20.

In [ ]:
# Your solution here


### Puzzle 13 — Compute Delta Metrics Across Periods

Given a DataFrame with a `date` column and `pm25` values, compute:
- This week's average
- Last week's average
- The delta (change)
- Whether air quality improved or worsened

Format as values suitable for `st.metric()`.

In [ ]:
# Your solution here


### Puzzle 14 — Build a Pivot Table for Display

Create a pivot table with counties as rows, months as columns, and mean PM2.5 as values.

Use `pd.pivot_table()`. This is the kind of table you would display with `st.dataframe()`.

In [ ]:
# Your solution here


### Puzzle 15 — Simulate `st.session_state` with a Dict

Streamlit uses `st.session_state` to persist values across reruns. Simulate this:

1. Create a `session_state` dict
2. Write a function `toggle_filter(session_state)` that flips a boolean `show_filter` key
3. Simulate 3 "reruns" and print the state each time

In [ ]:
# Your solution here


### Puzzle 16 — Write a Data Loading Function with Error Handling

Write `safe_load_csv(filepath)` that:
- Returns the DataFrame if the file exists
- Returns `None` and prints a user-friendly error if the file is missing
- Returns `None` and prints a message if the CSV is empty

This is the pattern you'd use with `st.error()` in a real app.

In [ ]:
# Your solution here


### Puzzle 17 — Generate Tab Labels from Data

Streamlit `st.tabs(labels)` creates tabbed panels. Given a DataFrame, dynamically generate tab labels from unique column values.

Example: one tab per county. Print the labels list and simulate assigning data to each tab.

In [ ]:
# Your solution here


### Puzzle 18 — Resample Time Series for a Chart

Given daily PM2.5 data for 90 days, resample to weekly and monthly averages. Print both aggregated DataFrames.

This is the data preparation you'd do before plotting a trend chart in Streamlit.

In [ ]:
# Your solution here


### Puzzle 19 — Build a Download Payload

Streamlit `st.download_button(data=...)` accepts a CSV string. Write a function `df_to_csv_string(df)` that converts a DataFrame to a CSV string suitable for download.

Test with a small DataFrame and print the CSV string.

In [ ]:
# Your solution here


### Puzzle 20 — Full Dashboard Data Pipeline *(Bonus)*

Simulate the complete data flow of a Streamlit dashboard:
1. Load a CSV file (create mock data first)
2. Parse dates and convert types
3. Apply sidebar filters (county, date range)
4. Compute summary metrics (mean, max, delta)
5. Group data for a bar chart
6. Resample for a trend line chart

Print each step's output to trace the full pipeline.

In [ ]:
# Your solution here
